In [0]:
--   DATA QUALITY CHECKS
--   A pipeline can look like it ran successfully but still contain
--   silent errors — wrong values, missing months, or rows that passed
--   the pipeline but should not have. These checks catch those problems.
--
--   Think of it like a pilot's pre-flight checklist. The plane might
--   look fine — but you check everything before taking off.
--
--   Check 1  — Bronze row counts
--   Check 2  — SST null check
--   Check 3  — SST range check
--   Check 4  — CC null check
--   Check 5  — CC monthly completeness (flags May 2024 gap)
--   Check 6  — Duplicate check
--   Check 7  — H3 null check across all Silver tables
--   Check 8  — Animal tracking completeness
--   Check 9  — Gold summary check
--   Check 10 — Final pipeline PASS/FAIL summary

In [0]:
-- Expected:
-- SST master    — roughly 89,000 rows per year across 11 years
-- CC master     — similar to SST
-- Manta Ray     — roughly 3,224 rows
-- Whale Shark   — roughly 3,382 rows
-- Sperm Whale   — roughly 7,759 rows
-- Green Turtle  — roughly 55,411 rows
-- Microplastics — 471 rows
-- ENSO          — one row per month from 1979 onwards

In [0]:
SELECT 'SST master' AS table_name, count(*) AS rows, count(DISTINCT source_year) AS years FROM mcphersonsharyn_bronze.bronze_sst_master UNION ALL
SELECT 'CC master' AS table_name, count(*) AS rows, count(DISTINCT source_year) AS years FROM mcphersonsharyn_bronze.bronze_cc_master UNION ALL
SELECT 'Manta Ray' AS table_name, count(*) AS rows, 0 AS years FROM mcphersonsharyn_bronze.bronze_manta_ray_raw UNION ALL
SELECT 'Whale Shark' AS table_name, count(*) AS rows, 0 AS years FROM mcphersonsharyn_bronze.bronze_whale_shark_raw UNION ALL
SELECT 'Sperm Whale' AS table_name, count(*) AS rows, 0 AS years FROM mcphersonsharyn_bronze.bronze_sperm_whale_raw UNION ALL
SELECT 'Green Turtle' AS table_name, count(*) AS rows, 0 AS years FROM mcphersonsharyn_bronze.bronze_green_turtle_raw UNION ALL
SELECT 'Microplastics' AS table_name, count(*) AS rows, 0 AS years FROM mcphersonsharyn_bronze.bronze_microplastics_raw UNION ALL
SELECT 'ENSO' AS table_name, count(*) AS rows, 0 AS years FROM mcphersonsharyn_bronze.bronze_enso_raw;

In [0]:
-- SST NULL CHECK                                     
-- All counts should be ZERO

SELECT
count(*) filter(WHERE sea_surface_temperature IS NULL) AS null_sst,
count(*) filter(WHERE latitude IS NULL) AS null_latitude,
count(*) filter(WHERE longitude IS NULL) AS null_longitude,
count(*) filter(WHERE year IS NULL) AS null_year,
count(*) filter(WHERE month IS NULL) AS null_month,
count(*) filter(WHERE h3_res4 IS NULL) AS null_h3_res4
FROM mcphersonsharyn_silver.sst_silver;

In [0]:
-- SST RANGE CHECK                                        
--  Should return ZERO rows                                         
--  If rows appear there are bad temperature values in Silver  

SELECT count(*) AS out_of_range_rows
FROM mcphersonsharyn_silver.sst_silver
WHERE sea_surface_temperature < -2
OR sea_surface_temperature > 40;

In [0]:
-- CC NULL CHECK                                      
-- null_latitude, null_longitude, null_h3 should be ZERO     
-- null_cc will be greater than zero — this is expected   

SELECT
count(*) filter(WHERE chlorophyll_concentration IS NULL) AS null_cc,
count(*) filter(WHERE latitude IS NULL) AS null_latitude,
count(*) filter(WHERE longitude IS NULL) AS null_longitude,
count(*) filter(WHERE h3_res4 IS NULL) AS null_h3
FROM mcphersonsharyn_silver.cc_silver;

In [0]:
-- CC MONTHLY COMPLETENESS                       
-- May 2024 should show SOURCE GAP — all other months should show OK 

SELECT
year,
month,
count(*) AS rows,
CASE
WHEN count(*) = 0 THEN 'SOURCE GAP — no NOAA data available'
ELSE 'OK'
END AS status
FROM mcphersonsharyn_silver.cc_silver
GROUP BY year, month
ORDER BY year, month;

In [0]:
-- DUPLICATE CHECK                                      
-- Should return ZERO                                       
-- If duplicates exist they will inflate row counts in Gold   

SELECT count(*) AS duplicate_combinations
FROM (
SELECT h3_res4, year, month, count(*) AS cnt
FROM mcphersonsharyn_silver.environmental_conditions_silver
GROUP BY h3_res4, year, month
HAVING count(*) > 1
);

In [0]:
-- H3 NULL CHECK ACROSS ALL SILVER TABLES                
--  null_h3 should be ZERO for every table                     
--  A null H3 cell means that row cannot be spatially joined   



SELECT 'sst_silver' AS table_name, count(*) filter(WHERE h3_res4 IS NULL) AS null_h3 FROM mcphersonsharyn_silver.sst_silver UNION ALL
SELECT 'cc_silver' AS table_name, count(*) filter(WHERE h3_res4 IS NULL) AS null_h3 FROM mcphersonsharyn_silver.cc_silver UNION ALL
SELECT 'microplastics_silver' AS table_name, count(*) filter(WHERE h3_res4 IS NULL) AS null_h3 FROM mcphersonsharyn_silver.microplastics_silver UNION ALL
SELECT 'manta_ray_silver' AS table_name, count(*) filter(WHERE h3_res4 IS NULL) AS null_h3 FROM mcphersonsharyn_silver.manta_ray_silver UNION ALL
SELECT 'whale_shark_silver' AS table_name, count(*) filter(WHERE h3_res4 IS NULL) AS null_h3 FROM mcphersonsharyn_silver.whale_shark_silver UNION ALL
SELECT 'sperm_whale_silver' AS table_name, count(*) filter(WHERE h3_res4 IS NULL) AS null_h3 FROM mcphersonsharyn_silver.sperm_whale_silver UNION ALL
SELECT 'green_turtle_silver' AS table_name, count(*) filter(WHERE h3_res4 IS NULL) AS null_h3 FROM mcphersonsharyn_silver.green_turtle_silver;

In [0]:
-- Animal Tracking Completness
-- Expected:
-- Reef Manta Ray  — 25 animals, 2014-2022
-- Whale Shark     — 41 animals, 2009-2015
-- Sperm Whale     — 17 animals, 2011-2013
-- Green Turtle    — 33 animals, 2012-2019
 
SELECT
species_common,
count(*) AS total_observations,
count(DISTINCT animal_id) AS unique_animals,
min(year) AS earliest_year,
max(year) AS latest_year,
count(DISTINCT season) AS seasons_covered,
count(DISTINCT locality) AS localities
FROM mcphersonsharyn_silver.migration_silver
GROUP BY species_common
ORDER BY species_common;

In [0]:
-- Gold Summary Check
-- null_sst and null_h3 must be zero
-- min_sst should be around 15 degrees C
-- max_sst should be below 35 degrees C
 
SELECT
count(*) AS total_rows,
count(*) filter(WHERE sst IS NULL) AS null_sst,
count(*) filter(WHERE h3_res4 IS NULL) AS null_h3,
round(min(sst), 2) AS min_sst,
round(max(sst), 2) AS max_sst,
round(avg(sst), 2) AS avg_sst,
count(DISTINCT year) AS years_loaded,
count(DISTINCT month) AS months_loaded
FROM mcphersonsharyn_gold.gold_environment_monthly;

In [0]:
-- FINAL PIPELINE PASS / FAIL SUMMARY                  
-- Run this at the end to see the health of the whole pipeline    
-- All rows should show PASS before running analysis  



SELECT
'Bronze SST' AS layer,
count(*) AS rows,
count(DISTINCT source_year) AS years,
CASE WHEN count(*) > 0 THEN 'PASS' ELSE 'FAIL — no rows found' END AS status
FROM mcphersonsharyn_bronze.bronze_sst_master
UNION ALL
SELECT
'Bronze CC',
count(*),
count(DISTINCT source_year),
CASE WHEN count(*) > 0 THEN 'PASS' ELSE 'FAIL — no rows found' END
FROM mcphersonsharyn_bronze.bronze_cc_master
UNION ALL
SELECT
'Silver SST',
count(*),
count(DISTINCT year),
CASE
WHEN count(*) filter(WHERE sea_surface_temperature IS NULL) > 0 THEN 'FAIL — null SST values'
WHEN count(*) filter(WHERE h3_res4 IS NULL) > 0 THEN 'FAIL — null H3 values'
WHEN count(*) > 0 THEN 'PASS'
ELSE 'FAIL — no rows found'
END
FROM mcphersonsharyn_silver.sst_silver
UNION ALL
SELECT
'Silver CC',
count(*),
count(DISTINCT year),
CASE
WHEN count(*) filter(WHERE h3_res4 IS NULL) > 0 THEN 'FAIL — null H3 values'
WHEN count(*) > 0 THEN 'PASS'
ELSE 'FAIL — no rows found'
END
FROM mcphersonsharyn_silver.cc_silver
UNION ALL
SELECT
'Silver ENSO',
count(*),
count(DISTINCT year),
CASE WHEN count(*) > 0 THEN 'PASS' ELSE 'FAIL — no rows found' END
FROM mcphersonsharyn_silver.enso_silver
UNION ALL
SELECT
'Environmental conditions Silver',
count(*),
count(DISTINCT year),
CASE
WHEN count(*) filter(WHERE sea_surface_temperature IS NULL) > 0 THEN 'FAIL — null SST values'
WHEN count(*) filter(WHERE h3_res4 IS NULL) > 0 THEN 'FAIL — null H3 values'
WHEN count(*) > 0 THEN 'PASS'
ELSE 'FAIL — no rows found'
END
FROM mcphersonsharyn_silver.environmental_conditions_silver
UNION ALL
SELECT
'Migration Silver — all species',
count(*),
count(DISTINCT year),
CASE
WHEN count(*) filter(WHERE h3_res4 IS NULL) > 0 THEN 'FAIL — null H3 values'
WHEN count(DISTINCT species_common) < 4 THEN 'FAIL — fewer than 4 species loaded'
WHEN count(*) > 0 THEN 'PASS'
ELSE 'FAIL — no rows found'
END
FROM mcphersonsharyn_silver.migration_silver
UNION ALL
SELECT
'Gold environment monthly',
count(*),
count(DISTINCT year),
CASE
WHEN count(*) filter(WHERE sst IS NULL) > 0 THEN 'FAIL — null SST values'
WHEN count(*) filter(WHERE h3_res4 IS NULL) > 0 THEN 'FAIL — null H3 values'
WHEN count(*) > 0 THEN 'PASS'
ELSE 'FAIL — no rows found'
END
FROM mcphersonsharyn_gold.gold_environment_monthly
UNION ALL
SELECT
'Gold migration joined',
count(*),
count(DISTINCT year),
CASE
WHEN count(DISTINCT species_common) < 4 THEN 'FAIL — fewer than 4 species'
WHEN count(*) > 0 THEN 'PASS'
ELSE 'FAIL — no rows found'
END
FROM mcphersonsharyn_gold.migration_environmental_joined;